# Step 1: Sequence Construction + Temporal Split
**Owner:** Bhakti Vipul Pasnani

This is the **shared prerequisite** step for the Baseline Team. It builds on
`DataIngestion_Subsample.ipynb` and produces the artifacts that Sonika's
candidate-sampling / popularity-baseline / SASRec notebook will load, so we
are not editing the same notebook at the same time.

**What this notebook does:**
1. Loads the active-user cohort's purchase data.
2. Builds chronological per-user purchase sequences, capped at `MAX_SEQ_LEN`.
3. Applies a **global temporal cutoff** (not per-user leave-one-out) to split
   into train / validation / test, so no future information leaks across users.
4. Saves everything Sonika needs to `artifacts/sequences_and_splits.pkl`.

**Before running this for real, confirm `MAX_SEQ_LEN`, `TEST_DAYS`, and
`VAL_DAYS` with the Cart-Aware team** -- these three values must be identical
across both subteams' pipelines or the final model comparison won't be valid.

In [ ]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 50)

## 0. Config

Confirm these three values with the Cart-Aware team before changing them.

In [ ]:
# ---- SHARED CONFIG: confirm with Cart-Aware team before changing ----
MAX_SEQ_LEN = 50   # based on our EDA: P95 sequence length = 47 events
TEST_DAYS = 14      # last 14 days of the observation window -> test
VAL_DAYS = 14       # the 14 days before that -> validation

CANDIDATE_DIRS = [
    Path("/content/subsampled"),
    Path("Desktop/DS5500/data/subsampled"),
    Path.home() / "Desktop" / "DS5500" / "data" / "subsampled",
]
DATA_DIR = next((p for p in CANDIDATE_DIRS if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find subsampled data. Run DataIngestion_Subsample.ipynb first, "
        f"or place parquet files under one of: {[str(p) for p in CANDIDATE_DIRS]}"
    )
print(f"Loading from: {DATA_DIR.resolve()}")

OUTPUT_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. Load purchase data

In [ ]:
product_buy = pd.read_parquet(DATA_DIR / "product_buy_subsampled.parquet")
product_buy = product_buy.drop_duplicates().copy()
product_buy["timestamp"] = pd.to_datetime(product_buy["timestamp"], errors="coerce")
product_buy = product_buy.dropna(subset=["timestamp"]).sort_values(["client_id", "timestamp"])

n_users = product_buy["client_id"].nunique()
n_events = len(product_buy)
min_ts, max_ts_display = product_buy["timestamp"].min(), product_buy["timestamp"].max()
print(f"Purchase events: {n_events:,} | Unique users: {n_users:,}")
print(f"Observation window: {min_ts} -> {max_ts_display}")

## 2. Global temporal split

A user's *train* sequence is every purchase strictly before `val_cutoff`. Their
*validation target* (if any) is their first purchase in `[val_cutoff, test_cutoff)`,
and their *test target* (if any) is their first purchase in `[test_cutoff, end]`.
Users with no purchase after a cutoff simply have no target for that split -- they
still contribute to training.

In [ ]:
max_ts = product_buy["timestamp"].max()
test_cutoff = max_ts - pd.Timedelta(days=TEST_DAYS)
val_cutoff = test_cutoff - pd.Timedelta(days=VAL_DAYS)

print(f"Train  : < {val_cutoff}")
print(f"Val    : [{val_cutoff}, {test_cutoff})")
print(f"Test   : [{test_cutoff}, {max_ts}]")

train_events = product_buy[product_buy["timestamp"] < val_cutoff]
val_events = product_buy[(product_buy["timestamp"] >= val_cutoff) & (product_buy["timestamp"] < test_cutoff)]
test_events = product_buy[product_buy["timestamp"] >= test_cutoff]

print(f"\nTrain events: {len(train_events):,}")
print(f"Val events  : {len(val_events):,}")
print(f"Test events : {len(test_events):,}")

## 3. Build per-user sequences and evaluation targets

In [ ]:
def build_sequences(events: pd.DataFrame, max_len: int) -> dict:
    """Return {client_id: [sku1, sku2, ...]} truncated to the last max_len items."""
    seqs = {}
    for client_id, group in events.groupby("client_id"):
        skus = group.sort_values("timestamp")["sku"].tolist()
        seqs[client_id] = skus[-max_len:]
    return seqs

def first_target_per_user(events: pd.DataFrame) -> dict:
    """Return {client_id: first_sku_in_window} using the earliest timestamp per user."""
    if len(events) == 0:
        return {}
    first_idx = events.groupby("client_id")["timestamp"].idxmin()
    first_events = events.loc[first_idx]
    return dict(zip(first_events["client_id"], first_events["sku"]))

train_sequences = build_sequences(train_events, MAX_SEQ_LEN)
val_targets = first_target_per_user(val_events)
test_targets = first_target_per_user(test_events)

val_eval_users = [u for u in val_targets if u in train_sequences and len(train_sequences[u]) > 0]
test_eval_users = [u for u in test_targets if u in train_sequences and len(train_sequences[u]) > 0]

print(f"Users with train sequence : {len(train_sequences):,}")
print(f"Users evaluable on val    : {len(val_eval_users):,}")
print(f"Users evaluable on test   : {len(test_eval_users):,}")

seq_lens = [len(s) for s in train_sequences.values()]
print(f"\nTrain sequence length -- mean: {np.mean(seq_lens):.2f}, "
      f"median: {np.median(seq_lens):.0f}, max: {max(seq_lens)}")

## 4. Item vocabulary

Built here (not by Sonika) so both of us use the exact same item-to-index mapping.

In [ ]:
all_items = sorted(product_buy["sku"].unique())
item_to_idx = {sku: i + 1 for i, sku in enumerate(all_items)}  # 0 reserved for padding
idx_to_item = {i: sku for sku, i in item_to_idx.items()}
n_items = len(all_items)
print(f"Unique items (vocabulary size): {n_items:,}")

## 5. Save artifacts for Sonika

Everything downstream (candidate sampling, popularity baseline, SASRec training/eval)
loads from this one file, so our two notebooks never need to touch each other's code.

In [ ]:
artifacts = {
    "config": {
        "MAX_SEQ_LEN": MAX_SEQ_LEN,
        "TEST_DAYS": TEST_DAYS,
        "VAL_DAYS": VAL_DAYS,
        "val_cutoff": val_cutoff,
        "test_cutoff": test_cutoff,
    },
    "train_sequences": train_sequences,
    "val_targets": val_targets,
    "test_targets": test_targets,
    "val_eval_users": val_eval_users,
    "test_eval_users": test_eval_users,
    "item_to_idx": item_to_idx,
    "idx_to_item": idx_to_item,
    "n_items": n_items,
    "all_items": all_items,
}

out_path = OUTPUT_DIR / "sequences_and_splits.pkl"
with open(out_path, "wb") as f:
    pickle.dump(artifacts, f)

print(f"Saved artifacts to: {out_path.resolve()}")
print(f"File size: {out_path.stat().st_size / 1024:.1f} KB")

## 6. Sanity checks before handing off

Quick checks so Sonika isn't debugging *my* pipeline inside *her* notebook if
something's off.

In [ ]:
# Every user with a val/test target must also have a train sequence
assert all(u in train_sequences for u in val_eval_users), "val_eval_users leak check failed"
assert all(u in train_sequences for u in test_eval_users), "test_eval_users leak check failed"

# No train sequence should exceed the configured max length
assert max(len(s) for s in train_sequences.values()) <= MAX_SEQ_LEN, "sequence length cap violated"

# Targets should never appear as duplicates of the item vocabulary issue (sanity on indexing)
assert all(sku in item_to_idx for sku in val_targets.values()), "val target missing from vocab"
assert all(sku in item_to_idx for sku in test_targets.values()), "test target missing from vocab"

print("All sanity checks passed. Safe to hand off to Sonika.")